In [1]:
import pandas as pd
import re
import string
import os
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer


In [2]:
# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
TRAIN_PATH   = '../data/raw/train.csv'
TEST_PATH    = '../data/raw/test.csv'
OUTPUT_DIR   = '../data/processed'
SAMPLE_SIZE  = None

os.makedirs(OUTPUT_DIR, exist_ok=True)


## Load Data

In [4]:
df = pd.read_csv(
    TRAIN_PATH,
    header=0,
    names=['label', 'text'],
    nrows=SAMPLE_SIZE
)

print(f'Shape   : {df.shape}')
df.head()


Shape   : (650000, 2)


,label,text
0,4,dr. goldberg offers everything i look for in a...
1,1,"Unfortunately, the frustration of being Dr. Go..."
2,3,Been going to Dr. Goldberg for over 10 years. ...
3,3,Got a letter in the mail last week that said D...
4,0,I don't know what Dr. Goldberg was like before...


## Data Cleaning

In [5]:
# Missing values
df.isnull().sum()

label    0
text     0
dtype: int64

In [6]:
# Duplicate rows
df.duplicated().sum()

0

In [7]:
#    Convert 5-class label to binary.
# Positive (1) = labels 3, 4  (4 & 5 stars)
# Negative (0) = labels 0, 1  (1 & 2 stars)
# Neutral  (2) = label  2     (3 stars)
def convert_to_binary_label(label: int) -> int:
    return 1 if label >= 3 else 0


## Preprocessing Pipeline

In [8]:
# Lowercase
def to_lowercase(text: str) -> str:
    return text.lower()

In [9]:
# Remove URLs
def remove_urls(text: str) -> str:
    return re.sub(r'https?://\S+|www\.\S+', '', text)

In [10]:
# Remove HTML tags
def remove_html_tags(text: str) -> str:
    return re.sub(r'<.*?>', '', text)

In [11]:
# Remove all punctuation characters
def remove_punctuation(text: str) -> str:
    return text.translate(str.maketrans('', '', string.punctuation))

In [12]:
# removes digits & symbols
def remove_special_characters(text: str) -> str:
    return re.sub(r'[^a-z\s]', '', text)

In [13]:
# Remove extra whitespace
def remove_extra_spaces(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()

In [14]:
# Split text into individual word tokens
def tokenize(text: str) -> list:
    return word_tokenize(text)

In [15]:
# Remove stopwords
STOP_WORDS  = set(stopwords.words('english'))
def remove_stopwords(tokens: list) -> list:
    return [t for t in tokens if t not in STOP_WORDS]

In [16]:
# Reduce each token to its base (dictionary) form
lemmatizer = WordNetLemmatizer()
def lemmatize(tokens: list) -> list:
    return [lemmatizer.lemmatize(t) for t in tokens]

In [17]:
def preprocess_text(text: str) -> str:
    text = to_lowercase(text)
    text = remove_urls(text)
    text = remove_html_tags(text)
    text = remove_punctuation(text)
    text = remove_special_characters(text)
    text = remove_extra_spaces(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize(tokens)
    return ' '.join(tokens)


In [18]:
# Drop neutral reviews
original_len = len(df)
df = df[df['label'] != 2].reset_index(drop=True)
print(f'Removed neutral reviews : {original_len - len(df):,}')
print(f'Remaining rows : {len(df):,}')

Removed neutral reviews : 130,000
Remaining rows : 520,000


In [19]:
df['binary_label'] = df['label'].apply(convert_to_binary_label)
df['cleaned_text'] = df['text'].apply(preprocess_text)


In [20]:
# Word count statistics
df['original_word_count'] = df['text'].apply(lambda x: len(x.split()))
df['cleaned_word_count']  = df['cleaned_text'].apply(lambda x: len(x.split()))

print('WORD COUNT STATISTICS')
print('\nOriginal text:')
print(df['original_word_count'].describe().round(1))
print('\nCleaned text:')
print(df['cleaned_word_count'].describe().round(1))

reduction = (1 - df['cleaned_word_count'].mean() / df['original_word_count'].mean()) * 100
print(f'\nAverage word reduction: {reduction:.1f}%')

WORD COUNT STATISTICS

Original text:
count    520000.0
mean        133.0
std         122.5
min           1.0
25%          51.0
50%          97.0
75%         173.0
max        1052.0
Name: original_word_count, dtype: float64

Cleaned text:
count    520000.0
mean         67.4
std          61.0
min           0.0
25%          27.0
50%          49.0
75%          88.0
max         822.0
Name: cleaned_word_count, dtype: float64

Average word reduction: 49.3%


In [21]:
# Drop rows where cleaned_text is empty
empty_cleaned = df['cleaned_text'].str.strip().eq('')
print(f'Rows with empty cleaned_text: {empty_cleaned.sum()}')

if empty_cleaned.sum() > 0:
    df = df[~empty_cleaned].reset_index(drop=True)
    print(f'Remaining rows: {len(df):,}')

Rows with empty cleaned_text: 44
Remaining rows: 519,956


In [22]:
# Before vs After preview
for i, row in df.sample(3, random_state=42).iterrows():
    print(f"\n[binary_label: {row['binary_label']}]")
    print(f"  Original : {row['text'][:200]}")
    print(f"  Cleaned  : {row['cleaned_text'][:200]}")


[binary_label: 0]
  Original : I've gone to this establishment two other times and had no issues. However there is a first time for everything. I stopped by today at around noon, ordered a Coconut Milk Tea with Boba. To be told the
  Cleaned  : ive gone establishment two time issue however first time everything stopped today around noon ordered coconut milk tea boba told run boba definitely last time go im still shocked place run boba hour o

[binary_label: 0]
  Original : We got the take out pad see ew tonight. It was not great. The typical pas see ew we've had has a sweet, brown, thick soy sauce. This was a very loose and spicy clear sauce.
  Cleaned  : got take pad see ew tonight great typical pa see ew weve sweet brown thick soy sauce loose spicy clear sauce

[binary_label: 0]
  Original : It angers me to be so hyped up about a place and then have to give them such a low grade rating. Overrated, overpriced and under serviced would be the top 3 ways I could describe it. I was so st

## Save Train Set

In [23]:
df_train_final = df[['binary_label', 'text', 'cleaned_text']].copy()

print(f'Shape   : {df_train_final.shape}')
df_train_final.head()

Shape   : (519956, 3)


,binary_label,text,cleaned_text
0,1,dr. goldberg offers everything i look for in a...,dr goldberg offer everything look general prac...
1,0,"Unfortunately, the frustration of being Dr. Go...",unfortunately frustration dr goldberg patient ...
2,1,Been going to Dr. Goldberg for over 10 years. ...,going dr goldberg year think one st patient st...
3,1,Got a letter in the mail last week that said D...,got letter mail last week said dr goldberg mov...
4,0,I don't know what Dr. Goldberg was like before...,dont know dr goldberg like moving arizona let ...


In [24]:
train_output_path = os.path.join(OUTPUT_DIR, 'preprocessed_train.csv')
df_train_final.to_csv(train_output_path, index=False)


In [25]:
# Train label distribution
print('Label distribution (Train set):')
dist = df_train_final['binary_label'].value_counts().sort_index()
total = len(df_train_final)

for label, count in dist.items():
    sentiment = 'Positive' if label == 1 else 'Negative'
    print(f'  {sentiment} ({label}) : {count:>7,} rows ({count/total*100:5.1f}%)')

print(f'\n  Total : {total:>7,} rows')

Label distribution (Train set):
  Negative (0) : 259,970 rows ( 50.0%)
  Positive (1) : 259,986 rows ( 50.0%)

  Total : 519,956 rows


## Apply Pipeline — Test Set

In [26]:
df_test = pd.read_csv(TEST_PATH, header=0, names=['label', 'text'])

# Drop neutral reviews
df_test = df_test[df_test['label'] != 2].reset_index(drop=True)

# Apply binary label + text preprocessing
df_test['binary_label'] = df_test['label'].apply(convert_to_binary_label)
df_test['cleaned_text'] = df_test['text'].apply(preprocess_text)

# Drop rows where cleaned_text is empty
empty_test = df_test['cleaned_text'].str.strip().eq('')
if empty_test.sum() > 0:
    df_test = df_test[~empty_test].reset_index(drop=True)

# Keep only needed columns
df_test_final = df_test[['binary_label', 'text', 'cleaned_text']].copy()

print(f'Test set ready : {len(df_test_final):,} rows')
df_test_final.head()

Test set ready : 39,998 rows


,binary_label,text,cleaned_text
0,0,I got 'new' tires from them and within two wee...,got new tire within two week got flat took car...
1,0,Don't waste your time. We had two different p...,dont waste time two different people come hous...
2,0,All I can say is the worst! We were the only 2...,say worst people place lunch place freezing lo...
3,0,I have been to this restaurant twice and was d...,restaurant twice disappointed time wont go bac...
4,0,Food was NOT GOOD at all! My husband & I ate h...,food good husband ate couple week ago first ti...


## Save Test Set

In [27]:
test_output_path = os.path.join(OUTPUT_DIR, 'preprocessed_test.csv')
df_test_final.to_csv(test_output_path, index=False)

In [28]:
# Test label distribution
print('Label distribution (Test set):')
dist = df_test_final['binary_label'].value_counts().sort_index()
total = len(df_test_final)

for label, count in dist.items():
    sentiment = 'Positive' if label == 1 else 'Negative'
    print(f'  {sentiment} ({label}) : {count:>7,} rows ({count/total*100:5.1f}%)')

print(f'\n  Total : {total:>7,} rows')

Label distribution (Test set):
  Negative (0) :  19,998 rows ( 50.0%)
  Positive (1) :  20,000 rows ( 50.0%)

  Total :  39,998 rows
